In [9]:
##########################
### notes on the stuff ###
##########################


# afaik: NLTK and WN are 2 separate WordNet readers ... so ... I only need one of them?
# depending on which one is better to work with maybe?
# does one of them have the possibility to read data that is not in their downloadable database?

In [10]:

########################
### console commands ###
### and imports etc. ###
########################

# console: pip install wn --upgrade
# --> wn
import wn
from nbformat.sign import yield_everything
from wn import Wordnet
# console: python -m wn download omw:1.4

# console: pip install pandas
import pandas as pd
import xml.etree.ElementTree as ET

In [11]:
# console: pip install nltk
# --> Natural Language Tool Kit
import nltk
from nltk.corpus import wordnet as nltkwn
nltk.download('omw-1.4')
nltk.download('wordnet')

[nltk_data] Downloading package omw-1.4 to C:\Users\E
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\E
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [83]:
#dir(nltkwn)
dir(nltkwn.synsets('ice'))
#nltkwn.synsets('ice')

['__add__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__delitem__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed__',
 '__rmul__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'append',
 'clear',
 'copy',
 'count',
 'extend',
 'index',
 'insert',
 'pop',
 'remove',
 'reverse',
 'sort']

In [69]:
id_list = nltkwn.synsets('ice')

tree = ET.parse(r'C:/Users/E T/Desktop/Studium/BA/code/wn_analysis/code/WordNets/HuWN/huwn.xml')
root = tree.getroot()

synsetlist = []

for e in root.findall('SYNSET'):
    for syn in e.find('SYNONYM').findall('LITERAL'):
        if 'jég' in syn.text and e.find('ID3') != None:
            synsetlist.append(e.find('ID3').text)
        else: continue

#print(synsetlist)
#print('ENG30-15036638-n'.split('-')[1].lstrip().split('-')[0])


### so this next one is supposed to compare the 'plant' synset id list from PWN to
### the synset list of HuWN to see whether there's a match
for x in range(len(id_list)):
    for id in synsetlist:
        if str(id_list[x].offset()) in id:
            print(str(id_list[x].offset()))
            print(id)

### result: the ID3 is the one corresponding to the PWN version... that is good! means that we can work with it...

14915184
ENG30-14915184-n


In [35]:
### testing verification of the Mongolian WordNet / checking where the connections between MoWN and PWN are

id_list = nltkwn.synsets('relationship')
for x in range(len(id_list)):
    if str(id_list[x].offset()) == '13780719':
        print(str(id_list[x].offset()))
        print(id_list[x])
### result 1 for харилцаа 13780719, which google translates to "relationship": 13780719, Synset('relationship.n.01')

nltkwn.synset_from_pos_and_offset('n', 2827883)
### result 2 for тууз 2827883, which google translates to "band/ribbon": Synset('belt.n.01')
### meaning they likely use the PWN id / offset for their synsets, which is great!

13780719
Synset('relationship.n.01')


Synset('belt.n.01')

In [7]:


#######################
### TURKISH WORDNET ###
#######################



In [8]:
import re
from translate import Translator

In [10]:
# console: pip install NlpToolkit-WordNet
# --> turkish WordNet
# on PC: C:\\Users\\torto\\Documents\\GitHub\\omw-data\\.git\\TurkishWordNet-Py\\WordNet\\data\\turkish_wordnet.xml
# on LT: C:\\Users\\E T\\TurkishWordNet-Py\\WordNet\\data\\turkish_wordnet.xml

from WordNet.WordNet import WordNet as kewn
twn = kewn("C:\\Users\\E T\\TurkishWordNet-Py\\WordNet\\data\\turkish_wordnet.xml")
ewn = kewn("C:\\Users\\E T\\TurkishWordNet-Py\\WordNet\\data\\english_wordnet_version_31.xml")

# trying to bring the synsets as told on the KeNet gitHub

tur_synsets = kewn.synSetList(twn)
en_synsets = kewn.synSetList(ewn)

In [26]:
#dir(ewn)
#dir(tur_synsets[0])
#dir(tur_synsets[0]._SynSet__relations)

# the following even gives us specifically the English mapped Synsets...
twn._WordNet__interlingual_list
relDict = twn._WordNet__interlingual_list
#relDict
#relDict['ENG31-14869913-n'] # this gives us Synset related to the Eng Synset
entry = relDict['ENG31-14869913-n'].__str__()
entry = entry.split(' ', 1)[0].replace('[', '')
entry # TUR ID for the Turkish Wordnet...
#twn.getSynSetWithId(entry) # getting entry from Tur WN


### that means... if we have the English ID, we can 'easily' access the Turkish Synset and all the relevant data...

#print(type(twn._WordNet__interlingual_list))

#twnDict = twn._WordNet__syn_set_list # this doesn't give us Eng rel tho
#twnDict['TUR10-0000030']


KeyError: slice(0, 1, 100)

In [19]:
### altered version of code below: print synset if there is interlingual info

for syn in tur_synsets[0:1000]:
    if len(syn.getInterlingual()) == 0:
        #print('no interlingual information\n')
        continue
    else: # this gives us the specific corresponding English Synsets
        print(syn._SynSet__id) # TUR ID of Synset
        print(f'\t{syn._SynSet__synonym}')# this gives us (all??) the literals in the Synset
        if len(syn._SynSet__relations) != 0:
            for rel in syn._SynSet__relations:
                r = rel.__str__()
                if r.split('->')[0] == "HYPERNYM" or r.split('->')[0] == "HYPONYM":
                    print(f'\t{r}')
        for id in syn.getInterlingual():
            print(ewn.getSynSetWithId(id))
        print('\n')

    #print(f'\t{syn._SynSet__relations}') # through this we can visualize all relations



TUR10-0000030
	su ab âb 
	HYPONYM->TUR10-0516740
	HYPONYM->TUR10-0678180
	HYPONYM->TUR10-0070540
	HYPONYM->TUR10-0418320
	HYPONYM->TUR10-0122950
	HYPONYM->TUR10-0194450
	HYPONYM->TUR10-0878120
	HYPONYM->TUR10-1241350
	HYPERNYM->TUR10-0530340
	HYPERNYM->TUR10-1239720
	HYPONYM->TUR10-0013110
	HYPONYM->TUR10-0629280
	HYPONYM->TUR10-0498990
	HYPONYM->TUR10-0393820
	HYPONYM->TUR10-0446630
	HYPONYM->TUR10-0476730
	HYPONYM->TUR10-0462630
	HYPONYM->TUR10-0981990
	HYPONYM->TUR10-1253790
	HYPONYM->TUR10-1125680
	HYPONYM->TUR10-0863040
	HYPONYM->TUR10-0849970
	HYPONYM->TUR10-0079110
	HYPONYM->TUR10-0429590
	HYPONYM->TUR10-0434300
	HYPONYM->TUR10-0534110
	HYPONYM->TUR10-0570460
	HYPONYM->TUR10-0588210
	HYPONYM->TUR10-0680300
	HYPONYM->TUR10-0711240
	HYPONYM->TUR10-0347760
	HYPONYM->TUR10-0169590
binary compound that occurs at room temperature as a clear colorless odorless tasteless liquid


TUR10-0000040
	kız kardeş kızkardeş bacı hemşire şvester 
	HYPERNYM->TUR10-1232030
	HYPONYM->TUR10-1232620
a

In [15]:
### this is rough code to extract synsets from the Turkish WordNet
### and to extract their corresponding English Synsets

for syn in tur_synsets[0:1000]:
    print(syn._SynSet__id) # TUR ID of Synset
    print(f'\t{syn._SynSet__synonym}')# this gives us (all??) the literals in the Synset
    if len(syn._SynSet__relations) != 0:
        for rel in syn._SynSet__relations:
            r = rel.__str__()
            if r.split('->')[0] == "HYPERNYM" or r.split('->')[0] == "HYPONYM":
                print(f'\t{r}')

    #print(f'\t{syn._SynSet__relations}') # through this we can visualize all relations

    if len(syn.getInterlingual()) == 0:
        print('no interlingual information\n')
        continue
    else: # this gives us the specific corresponding English Synsets
        for id in syn.getInterlingual():
            print(ewn.getSynSetWithId(id))
        print('\n')
#dir(tur_synsets)

#dir(tur_synsets[0])

#count = 0
#for synset in tur_synsets:
#    if dir(tur_synsets[0]) == dir(synset):
#        count += 1
#        continue
#    else: print(f'{synset} is not the same')
#print(count)

#translator = Translator(to_lang="en")

#print(type(tur_synsets[673]))

#print(type(tur_synsets))
#print(type(tur_synsets[0]))
#for w in range(0,100):
#    print(f'tu: {tur_synsets[w]} \n {type(tur_synsets[w])}')
#    try:
#        tr = translator.translate(str(tur_synsets[w]))
#        print(tr)
#    except:
#        print("translation error")
#        continue


### trying to extract all the literals from the WordNet... ###
#tur_list = tur_synsets.__str__().split(',')
#str(tur_list)
#res = re.findall(r"\((.*?)\)", str(tur_list))
#for w in res:
#    print(w)
#kewn.getSynSetWithLiteral(twn, literal=res[0], sense=0)

TUR10-0000000
	(özel isim) 
	HYPERNYM->TUR10-0379560
no interlingual information

TUR10-0000003
	(zaman) 
	HYPERNYM->TUR10-0869550
no interlingual information

TUR10-0000004
	(tarih) 
	HYPERNYM->TUR10-0747200
no interlingual information

TUR10-0000006
	(hashtag) 
	HYPERNYM->TUR10-0579300
no interlingual information

TUR10-0000007
	(eposta) 
	HYPERNYM->TUR10-0844140
no interlingual information

TUR10-0000010
	(tam sayı) 
	HYPERNYM->TUR10-0008070
no interlingual information

TUR10-0000011
	(sayı sıra sıfatı) 
	HYPERNYM->TUR10-0684420
no interlingual information

TUR10-0000013
	(yüzde) 
	HYPERNYM->TUR10-0079390
no interlingual information

TUR10-0000015
	(kesir sayı) 
	HYPERNYM->TUR10-0444660
no interlingual information

TUR10-0000018
	(sayı aralığı) 
	HYPERNYM->TUR10-0212060
no interlingual information

TUR10-0000020
	(reel sayı) 
	HYPERNYM->TUR10-0292910
no interlingual information

TUR10-0000028
	aba vakti yaba yaba vakti aba 
no interlingual information

TUR10-0000030
	su ab âb 
	HYPO

In [ ]:


#########################
### MONGOLIAN WORDNET ###
#########################



In [4]:
#TRYING to load Mongolian WordNet
df = pd.read_csv("C:\\Users\\E T\\Desktop\\Studium\\BA\\code\\wn_analysis\\code\\WordNets\\MonWN\\wn-data-mon.tsv", sep="\t")

In [5]:
display(df)

,# Mongolian Wordnet (MonWN),mon,https://github.com/kbatsuren/monwn,CC BY 4.0
0,06177450-n,mon:lemma,авиа судлал,NaN
1,06177729-n,mon:lemma,үг зүй,NaN
2,06177729-n,mon:def 0,үгийн дуудлагын зөвшөөрөгдсөн тогтолцоо,NaN
3,00001740-n,mon:lemma,нэгж,NaN
4,06177923-n,mon:lemma,залгавар,NaN
...,...,...,...,...
42604,06176107-n,mon:def 0,өгүүлбэр дэх үгсийн хэл зүйн тогтолцоо,NaN
42605,13773539-n,mon:lemma,гэгээ,NaN
42606,13773539-n,mon:lemma,ул мөр,NaN
42607,06177033-n,mon:lemma,авиа зүй,NaN


In [ ]:


####################
### nltkwn tests ###
####################



In [10]:
love_en = nltkwn.synsets('love', lang='eng')
print(love_en[0].definition())

a strong positive emotion of regard and affection


In [9]:
bg = wn.Wordnet('omw-bg:1.4')
da = wn.Wordnet('omw-da:1.4')
print(type(wn.Wordnet))

Error: no lexicon found with lang=None and lexicon='omw-bg:1.4'

In [ ]:
nltkwn.lemmas('amore', lang='ita')

In [ ]:
print(nltkwn.synset('love.n.01').lemma_names('ita'))
print(nltkwn.synset('love.n.01').lemma_names('arb'))

In [ ]:
for x in nltkwn.langs():
    print(str(x))
    print(nltkwn.synset('cat.n.01').lemma_names(str(x)))

In [ ]:
# what am i dong here T_T
print(nltkwn.synset('dog.n.01').lemma_names('pol'))
print('\'' + nltkwn.synset('dog.n.01').lemma_names('pol')[1] + '\'')
print(nltkwn.lemmas('\'' + nltkwn.synset('dog.n.01').lemma_names('pol')[1] + '\'', lang='pol'))

In [ ]:
for x in nltkwn.langs():
    nltkwn.lemma('love.n.01.' + nltkwn.synset('love.n.01').lemma_names(str(x))[0]).synset()

In [ ]:
dog_ita = nltkwn.synsets('cane', lang='ita')
print(dog_ita[0].definition())  # Still returns the English definition
print(dog_ita[0].lemmas('ita'))

In [ ]:


####################
### multiwordnet ###
####################



In [ ]:
from multiwordnet.db import compile
#compile('english')
#compile('italian')
#compile('french')
#compile('hebrew')
#compile('portuguese')
#compile('spanish')
compile('japanese')
compile('common')

In [ ]:
from multiwordnet.wordnet import WordNet

EWN = WordNet('english')
IWN = WordNet('italian')
FWN = WordNet('french')
HWN = WordNet('hebrew')
PWN = WordNet('portuguese')
SWN = WordNet('spanish')


In [ ]:
test = 0
for synset in HWN.synsets:
    test += 1
    if test > 10:
        break
    else:
        print(synset)
        print('\n')

In [ ]:
# English synsets
count = 0
for synset in EWN.synsets:
    if synset is not None:
        count += 1
    else: next
count_EWN = count
# finds 44569 synsets even if when loading the WordNet it shows 44576

In [ ]:
count_EWN = sum(1 for synset in EWN.synsets)

In [ ]:
count_FWN = sum(1 for synset in FWN.synsets)

In [ ]:
count_IWN = sum(1 for synset in IWN.synsets)

In [ ]:
count_HWN = sum(1 for synset in HWN.synsets)
print(count_HWN)

In [ ]:
count_PWN = sum(1 for synset in PWN.synsets)

In [ ]:
count_SWN = sum(1 for synset in SWN.synsets)

In [ ]:
print(str(count_EWN) + "/102389 synsets found")
print(str(count_FWN) + "/54995 synsets found")
print(str(count_IWN) + "/38748 synsets found")
print(str(count_HWN) + "/5929 synsets found")
print(str(count_PWN) + "/44576 synsets found")
print(str(count_SWN) + "/81292 synsets found")

# 102382 synsets found, according to loading: 102389
# 54965 synsets found, according to loading: 54995
# 38741 synsets found, according to loading: 38748
# 5922 synsets found, according to loading: 5929
# 44569 synsets found, according to loading: 44576
# 80281 synsets found, according to loading: 81292

In [1]:
angry = EWN.get_lemma('angry', 'a')

NameError: name 'EWN' is not defined

In [ ]:
angry.synonyms